# RECAP on SmolVLA — Suffix-side Advantage Conditioning

This notebook implements RECAP-style advantage-conditioned fine-tuning on SmolVLA for LIBERO-Spatial.

**Assumes:**
- Modal volume mounted at `/mnt/rollouts`
- Rollout dataset at `/mnt/rollouts/rollouts_with_labels` (100 labeled BC rollouts)
- GPU available

**Does:**
1. Installs lerobot
2. Loads BC baseline + adds advantage-token module (suffix-side injection)
3. Loads existing rollout dataset
4. Downloads + filters expert dataset to LIBERO-Spatial
5. Mixes expert + rollouts, trains RECAP with advantage conditioning
6. Sanity-checks inference (A_null should match BC)
7. Full eval: A_pos vs A_neg vs A_null

Run cells top to bottom.


## Step 1 — Environment

In [13]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["HF_HOME"]         = "/mnt/rollouts/hf_cache"
os.environ["HF_LEROBOT_HOME"] = "/mnt/rollouts/lerobot_cache"


In [14]:
!apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!pip install -q "lerobot[smolvla,libero]"



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import torch
print(f"CUDA: {torch.cuda.is_available()}  |  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


CUDA: True  |  NVIDIA A100-SXM4-40GB


## Step 2 — LIBERO config (prevents import prompt crash)

In [16]:
import yaml
from pathlib import Path
import libero

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(parents=True, exist_ok=True)
libero_data_root = Path("/mnt/rollouts/libero_data")
libero_data_root.mkdir(parents=True, exist_ok=True)
pkg_dir = Path(libero.__file__).parent / "libero"
with open(libero_config_dir / "config.yaml", "w") as f:
    yaml.safe_dump({
        "benchmark_root": str(pkg_dir),
        "bddl_files":     str(pkg_dir / "bddl_files"),
        "init_states":    str(pkg_dir / "init_files"),
        "datasets":       str(libero_data_root),
        "assets":         str(pkg_dir / "assets"),
    }, f)
print("LIBERO config done.")


LIBERO config done.


## Step 3 — Constants

In [17]:
from pathlib import Path
import numpy as np

DRIVE_ROOT          = "/mnt/rollouts"
ROLLOUT_DATASET_DIR = Path(DRIVE_ROOT) / "rollouts_with_labels"
RECAP_V2_DIR        = Path(DRIVE_ROOT) / "recap_v2_suffix"
RECAP_V2_DIR.mkdir(parents=True, exist_ok=True)

BC_POLICY_REPO  = "HuggingFaceVLA/smolvla_libero"
LIBERO_SUITE    = "libero_spatial"
ADV_NEG, ADV_POS, ADV_NULL = 0, 1, 2

TASK_DESCRIPTIONS = {
    0: "pick up the black bowl between the plate and the ramekin and place it on the plate",
    1: "pick up the black bowl from table center and place it on the plate",
    2: "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    3: "pick up the black bowl next to the cookie box and place it on the plate",
    4: "pick up the black bowl next to the plate and place it on the plate",
    5: "pick up the black bowl next to the ramekin and place it on the plate",
    6: "pick up the black bowl on the cookie box and place it on the plate",
    7: "pick up the black bowl on the ramekin and place it on the plate",
    8: "pick up the black bowl on the stove and place it on the plate",
    9: "pick up the black bowl on the wooden cabinet and place it on the plate",
}

# Verify rollout data is accessible
assert ROLLOUT_DATASET_DIR.exists(), f"Rollout dataset not found at {ROLLOUT_DATASET_DIR}"
print(f"ROLLOUT_DATASET_DIR = {ROLLOUT_DATASET_DIR} (exists)")
print(f"RECAP_V2_DIR        = {RECAP_V2_DIR}")


ROLLOUT_DATASET_DIR = /mnt/rollouts/rollouts_with_labels (exists)
RECAP_V2_DIR        = /mnt/rollouts/recap_v2_suffix


## Step 4 — Advantage token module

Unlike the first RECAP attempt, this version adds the advantage embedding to the **suffix** (noisy action tokens),
not the prefix (VLM features). The suffix is rebuilt every denoising step so it bypasses the KV-cache
contamination that broke our earlier prefix-based injection.


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SuffixAdvantageEmbedding(nn.Module):
    """Advantage embedding sized to the action expert's hidden dim.
    Added to every suffix token at every denoise step."""
    def __init__(self, expert_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(3, expert_hidden_size)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.embedding.weight[ADV_NULL].zero_()  # null = zero vector

    def forward(self, advantage_labels):
        return self.embedding(advantage_labels)


def _patched_embed_suffix(self, noisy_actions, timestep):
    """Patched embed_suffix that adds the advantage embedding to every action-time token."""
    embs, pad_masks, att_masks = self.__class__._orig_embed_suffix(self, noisy_actions, timestep)
    label = getattr(self, "_train_advantage_label",
                    getattr(self, "_eval_advantage_label", ADV_POS))
    bs = embs.shape[0]
    if isinstance(label, torch.Tensor):
        adv_labels = label.to(embs.device)
    else:
        adv_labels = torch.full((bs,), int(label), dtype=torch.long, device=embs.device)
    adv_emb = self.advantage_token(adv_labels)           # (B, expert_hidden)
    embs = embs + adv_emb[:, None, :].to(embs.dtype)     # broadcast add across chunk positions
    return embs, pad_masks, att_masks


def set_eval_advantage(pol, label):
    pol.model._eval_advantage_label = int(label)


print("Suffix-side advantage token module defined.")


Suffix-side advantage token module defined.


## Step 5 — Load BC, attach advantage token, freeze, apply patch

In [19]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks

device = torch.device("cuda")
print(f"Loading BC baseline from {BC_POLICY_REPO} ...")
policy = SmolVLAPolicy.from_pretrained(BC_POLICY_REPO).to(device)

expert_hidden = policy.model.vlm_with_expert.expert_hidden_size
print(f"expert_hidden_size = {expert_hidden}")

policy.model.advantage_token = SuffixAdvantageEmbedding(expert_hidden).to(device)

cls = type(policy.model)
if not hasattr(cls, "_orig_embed_suffix"):
    cls._orig_embed_suffix = cls.embed_suffix
cls.embed_suffix = _patched_embed_suffix

TRAINABLE_KEYWORDS = (
    "advantage_token",
    "action_in_proj", "action_out_proj",
    "action_time_mlp",
    "state_proj",
    "lm_expert",
)
for name, p in policy.named_parameters():
    p.requires_grad = any(k in name for k in TRAINABLE_KEYWORDS)

trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
total     = sum(p.numel() for p in policy.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M")

tokenizer = policy.model.vlm_with_expert.processor.tokenizer
CHUNK_SIZE = policy.config.chunk_size
MAX_ACTION_DIM = policy.config.max_action_dim
print(f"CHUNK_SIZE={CHUNK_SIZE}, MAX_ACTION_DIM={MAX_ACTION_DIM}")


Loading BC baseline from HuggingFaceVLA/smolvla_libero ...
Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

expert_hidden_size = 480
Trainable: 97.5M / 604.9M
CHUNK_SIZE=50, MAX_ACTION_DIM=32


## Step 6 — Rollout dataset loader (reads existing parquet)

In [20]:
import glob
import pandas as pd
from torch.utils.data import Dataset


def _unpack(x):
    """Recursively flatten nested list-of-lists (parquet structure) into flat floats."""
    out = []
    stack = [x]
    while stack:
        item = stack.pop()
        if isinstance(item, (list, tuple, np.ndarray)):
            stack.extend(reversed(list(item)) if hasattr(item, "__iter__") else [item])
        else:
            out.append(item)
    return out


class RolloutFrameDataset(Dataset):
    def __init__(self, root):
        self.files = sorted(glob.glob(str(Path(root) / "data" / "chunk-000" / "file-*.parquet")))
        assert self.files, f"No parquet files in {root}/data/chunk-000/"

        self._labels = []
        self._file_idx = []
        self._row_idx = []
        self._task_idx = []

        tasks_pq = Path(root) / "meta" / "tasks.parquet"
        tdf = pd.read_parquet(tasks_pq)
        self._tasks_map = dict(zip(tdf["task_index"].tolist(), tdf.index.tolist()))

        print(f"Indexing {len(self.files)} parquet files ...")
        for fi, f in enumerate(self.files):
            df = pd.read_parquet(f, columns=["advantage_label", "task_index"])
            n = len(df)
            for r in range(n):
                lbl_raw = df["advantage_label"].iat[r]
                ti_raw = df["task_index"].iat[r]
                self._labels.append(int(lbl_raw[0]) if hasattr(lbl_raw, "__len__") else int(lbl_raw))
                self._task_idx.append(int(ti_raw[0]) if hasattr(ti_raw, "__len__") else int(ti_raw))
                self._file_idx.append(fi)
                self._row_idx.append(r)

        self._labels = np.array(self._labels, dtype=np.int64)
        self._task_idx = np.array(self._task_idx, dtype=np.int64)
        self._cache = {}
        print(f"Rollout frames: {len(self._labels)}  "
              f"(pos={(self._labels==1).sum()}, neg={(self._labels==0).sum()})")

    def __len__(self):
        return len(self._labels)

    def _load_file(self, fi):
        if fi in self._cache:
            return self._cache[fi]
        df = pd.read_parquet(self.files[fi],
                             columns=["observation.images.image",
                                      "observation.images.image2",
                                      "observation.state",
                                      "action"])
        if len(self._cache) >= 4:
            self._cache.pop(next(iter(self._cache)))
        self._cache[fi] = df
        return df

    def __getitem__(self, i):
        df = self._load_file(self._file_idx[i])
        r = self._row_idx[i]
        img1  = np.asarray(_unpack(df["observation.images.image"].iat[r]),  dtype=np.float32).reshape(3, 256, 256)
        img2  = np.asarray(_unpack(df["observation.images.image2"].iat[r]), dtype=np.float32).reshape(3, 256, 256)
        state = np.asarray(_unpack(df["observation.state"].iat[r]), dtype=np.float32)
        action = np.asarray(_unpack(df["action"].iat[r]), dtype=np.float32)
        return {
            "observation.images.image":  torch.from_numpy(img1),
            "observation.images.image2": torch.from_numpy(img2),
            "observation.state":         torch.from_numpy(state),
            "action":                    torch.from_numpy(action),
            "advantage_label":           int(self._labels[i]),
            "task":                      self._tasks_map.get(int(self._task_idx[i]), ""),
        }


rollout_ds = RolloutFrameDataset(ROLLOUT_DATASET_DIR)


Indexing 47 parquet files ...
Rollout frames: 15462  (pos=7875, neg=7587)


## Step 7 — Expert dataset (download + filter to Spatial)

In [21]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("Loading expert dataset (will download if not cached) ...")
expert_ds = LeRobotDataset("HuggingFaceVLA/libero")
print(f"Expert dataset: {expert_ds.meta.total_episodes} episodes, {len(expert_ds)} frames")

# Filter to LIBERO-Spatial
SPATIAL_TASK_STRINGS = set(TASK_DESCRIPTIONS.values())
spatial_episodes = []
for ep_i in range(expert_ds.meta.total_episodes):
    ep = expert_ds.meta.episodes[ep_i]
    tasks = ep["tasks"]
    if len(tasks) == 1 and tasks[0] in SPATIAL_TASK_STRINGS:
        spatial_episodes.append((ep_i, ep["dataset_from_index"], ep["dataset_to_index"], tasks[0]))

print(f"Spatial episodes: {len(spatial_episodes)}")
total_spatial_frames = sum(dst - src for _, src, dst, _ in spatial_episodes)
print(f"Spatial frames: {total_spatial_frames}")


class ExpertSpatialDataset(Dataset):
    def __init__(self, ds, spatial_episodes):
        self.ds = ds
        self.frame_map = []
        for _, src, dst, task_str in spatial_episodes:
            for f in range(src, dst):
                self.frame_map.append((f, task_str))

    def __len__(self):
        return len(self.frame_map)

    def __getitem__(self, i):
        ds_idx, task_str = self.frame_map[i]
        s = self.ds[ds_idx]
        return {
            "observation.images.image":  s["observation.images.image"],
            "observation.images.image2": s["observation.images.image2"],
            "observation.state":         s["observation.state"],
            "action":                    s["action"],
            "advantage_label":           1,       # expert = always success
            "task":                      task_str,
        }


expert_spatial_ds = ExpertSpatialDataset(expert_ds, spatial_episodes)
print(f"Expert Spatial dataset: {len(expert_spatial_ds)} frames")


Loading expert dataset (will download if not cached) ...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Expert dataset: 1693 episodes, 273465 frames
Spatial episodes: 432
Spatial frames: 52970
Expert Spatial dataset: 52970 frames


## Step 8 — Mixed dataset + weighted sampler

In [22]:
from torch.utils.data import ConcatDataset, WeightedRandomSampler

mixed_ds = ConcatDataset([expert_spatial_ds, rollout_ds])

n_expert = len(expert_spatial_ds)
n_rollout_pos = int((rollout_ds._labels == 1).sum())
n_rollout_neg = int((rollout_ds._labels == 0).sum())
total_pos = n_expert + n_rollout_pos
total_neg = n_rollout_neg
print(f"Mix: expert={n_expert}  rollout_pos={n_rollout_pos}  rollout_neg={n_rollout_neg}")
print(f"Balance target: 50/50 (current raw ratio {total_pos/total_neg:.1f}x more pos than neg)")

# Per-sample weights: 1/N_class makes each class contribute equally in expectation
w_pos = 1.0 / total_pos
w_neg = 1.0 / total_neg
weights = np.empty(len(mixed_ds), dtype=np.float64)
weights[:n_expert] = w_pos
for i in range(len(rollout_ds)):
    weights[n_expert + i] = w_pos if rollout_ds._labels[i] == 1 else w_neg

sampler = WeightedRandomSampler(weights=weights, num_samples=len(mixed_ds), replacement=True)
print(f"Sampler ready. Mixed dataset size: {len(mixed_ds)}")


Mix: expert=52970  rollout_pos=7875  rollout_neg=7587
Balance target: 50/50 (current raw ratio 8.0x more pos than neg)
Sampler ready. Mixed dataset size: 68432


## Step 9 — Training loop

The advantage embedding is injected into the suffix via the patched `embed_suffix`. At each training
step we set `policy.model._train_advantage_label = batch_labels` so the patch injects the correct
per-sample advantage.

Watch the logs for `Δ=pos-neg` — it should grow more negative over training, meaning the model
learns to distinguish positive (success) from negative (failure) trajectories.


In [23]:
import json, time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from lerobot.utils.constants import ACTION


def collate(batch_list):
    return {
        "observation.images.image":  torch.stack([b["observation.images.image"] for b in batch_list]),
        "observation.images.image2": torch.stack([b["observation.images.image2"] for b in batch_list]),
        "observation.state":         torch.stack([b["observation.state"] for b in batch_list]),
        "action":                    torch.stack([b["action"] for b in batch_list]),
        "advantage_label":           torch.tensor([int(b["advantage_label"]) for b in batch_list], dtype=torch.long),
        "task":                      [b["task"] for b in batch_list],
    }


def batch_to_device(batch, dev):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(dev, non_blocking=True)
    return batch


def recap_loss_suffix(policy, batch, adv):
    bs = batch[ACTION].shape[0]
    dev = adv.device

    images, img_masks = policy.prepare_images(batch)
    state = policy.prepare_state(batch)
    actions = policy.prepare_action(batch)

    tok = tokenizer(list(batch["task"]), padding="max_length",
                    max_length=policy.config.tokenizer_max_length,
                    truncation=True, return_tensors="pt")
    lang_tokens = tok["input_ids"].to(dev)
    lang_masks = tok["attention_mask"].to(dev).bool()

    if actions.ndim == 2:
        actions = actions.unsqueeze(1).expand(-1, CHUNK_SIZE, -1)
    if actions.shape[-1] < MAX_ACTION_DIM:
        actions = F.pad(actions, (0, MAX_ACTION_DIM - actions.shape[-1]))

    noise = policy.model.sample_noise(actions.shape, dev)
    t = policy.model.sample_time(bs, dev)
    x_t = t[:, None, None] * noise + (1.0 - t[:, None, None]) * actions
    u_t = noise - actions

    # Tell the patched embed_suffix which labels to use this forward
    policy.model._train_advantage_label = adv

    prefix_embs, prefix_pad, _ = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state)
    suffix_embs, suffix_pad, _ = policy.model.embed_suffix(x_t, t)

    pad = torch.cat([prefix_pad, suffix_pad], dim=1)
    p_len, s_len = prefix_pad.shape[1], suffix_pad.shape[1]
    att = torch.cat([torch.zeros(bs, p_len, dtype=torch.bool, device=dev),
                     torch.ones(bs, s_len, dtype=torch.bool, device=dev)], dim=1)
    att_2d = make_att_2d_masks(pad, att)
    pos_ids = torch.cumsum(pad, dim=1) - 1

    (_, suffix_out), _ = policy.model.vlm_with_expert.forward(
        attention_mask=att_2d, position_ids=pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, suffix_embs],
        use_cache=False, fill_kv_cache=False)
    suffix_out = suffix_out[:, -CHUNK_SIZE:].to(dtype=torch.float32)
    v_t = policy.model.action_out_proj(suffix_out)

    real_dim = policy.config.action_feature.shape[0]
    per = F.mse_loss(u_t[..., :real_dim], v_t[..., :real_dim], reduction="none")
    return per.flatten(1).mean(dim=1)


STEPS      = 8_000
BATCH_SIZE = 16
LR         = 1e-4
SAVE_EVERY = 2_000
LOG_EVERY  = 50

dl = DataLoader(mixed_ds, batch_size=BATCH_SIZE, sampler=sampler,
                num_workers=4, pin_memory=True, drop_last=True,
                collate_fn=collate, persistent_workers=True, prefetch_factor=2)

optimizer = torch.optim.AdamW(
    [p for p in policy.parameters() if p.requires_grad],
    lr=LR, weight_decay=1e-10, betas=(0.9, 0.95),
)

policy.train()
history = []
running_tot = running_pos = running_neg = 0.0
n_pos_b = n_neg_b = 0
step = 0
t0 = time.time()
pbar = tqdm(total=STEPS, desc="training", dynamic_ncols=True)

while step < STEPS:
    for batch in dl:
        if step >= STEPS:
            break
        batch = batch_to_device(batch, device)
        adv = batch["advantage_label"].to(device).long()

        per_sample = recap_loss_suffix(policy, batch, adv)
        loss = per_sample.mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()

        with torch.no_grad():
            pm = (adv == 1); nm = (adv == 0)
            if pm.any():
                running_pos += per_sample[pm].mean().item(); n_pos_b += 1
            if nm.any():
                running_neg += per_sample[nm].mean().item(); n_neg_b += 1
            running_tot += loss.item()

        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{loss.item():.3f}"})

        if step % LOG_EVERY == 0:
            p = running_pos / max(n_pos_b, 1)
            n = running_neg / max(n_neg_b, 1)
            t = running_tot / LOG_EVERY
            tqdm.write(f"[step {step:5d}] loss={t:.4f} pos={p:.4f} neg={n:.4f} "
                       f"Δ={p-n:+.4f} elapsed={(time.time()-t0)/60:.1f}m")
            history.append((step, t, p, n))
            running_tot = running_pos = running_neg = 0.0
            n_pos_b = n_neg_b = 0

        if step % SAVE_EVERY == 0:
            ck = RECAP_V2_DIR / f"step_{step:06d}"
            ck.mkdir(parents=True, exist_ok=True)
            policy.save_pretrained(ck)

pbar.close()
policy.save_pretrained(RECAP_V2_DIR)
with open(RECAP_V2_DIR / "loss_history.json", "w") as f:
    json.dump(history, f)
print(f"\nTraining done. Saved to {RECAP_V2_DIR}")

training:   0%|                                                                       | 0/8000 [00:00<?, ?it/s…

[step    50] loss=0.1784 pos=0.1884 neg=0.1695 Δ=+0.0189 elapsed=2.2m
[step   100] loss=0.1226 pos=0.1210 neg=0.1223 Δ=-0.0013 elapsed=4.2m
[step   150] loss=0.1106 pos=0.1075 neg=0.1147 Δ=-0.0072 elapsed=6.3m
[step   200] loss=0.1040 pos=0.1048 neg=0.1021 Δ=+0.0028 elapsed=8.4m
[step   250] loss=0.0977 pos=0.0841 neg=0.1106 Δ=-0.0266 elapsed=10.7m
[step   300] loss=0.0867 pos=0.0783 neg=0.0941 Δ=-0.0158 elapsed=12.7m
[step   350] loss=0.0979 pos=0.0951 neg=0.0997 Δ=-0.0046 elapsed=14.9m
[step   400] loss=0.0914 pos=0.0856 neg=0.0973 Δ=-0.0117 elapsed=17.0m
[step   450] loss=0.0842 pos=0.0862 neg=0.0804 Δ=+0.0058 elapsed=19.2m
[step   500] loss=0.0832 pos=0.0766 neg=0.0916 Δ=-0.0150 elapsed=21.5m
[step   550] loss=0.0914 pos=0.0825 neg=0.0988 Δ=-0.0164 elapsed=23.7m
[step   600] loss=0.0870 pos=0.0810 neg=0.0903 Δ=-0.0092 elapsed=26.1m
[step   650] loss=0.0772 pos=0.0689 neg=0.0874 Δ=-0.0186 elapsed=28.3m
[step   700] loss=0.0733 pos=0.0665 neg=0.0793 Δ=-0.0128 elapsed=30.5m
[step   75

KeyboardInterrupt: 

## Step 10 — Training loss plot

In [ ]:
import matplotlib.pyplot as plt

hist = np.array(history)
plt.figure(figsize=(9, 4))
plt.plot(hist[:, 0], hist[:, 1], label="total", alpha=0.7)
plt.plot(hist[:, 0], hist[:, 2], label="pos (A=1)", alpha=0.7)
plt.plot(hist[:, 0], hist[:, 3], label="neg (A=0)", alpha=0.7)
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Suffix-RECAP training loss (pos vs neg)")
plt.tight_layout()
plt.savefig(RECAP_V2_DIR / "loss_curve.png", dpi=120)
plt.show()
print(f"Saved curve to {RECAP_V2_DIR / 'loss_curve.png'}")


## Step 11 — Set up env + preprocessors for eval

In [25]:
from copy import deepcopy
import einops
from tqdm import trange
import torch.nn as _nn
import lerobot.scripts.lerobot_eval as _le
from lerobot.envs.utils import preprocess_observation, add_envs_task, check_env_attributes_and_types
from lerobot.utils.constants import ACTION as _ACTION, OBS_STR
from lerobot.utils.utils import inside_slurm


def _patched_rollout(env, policy, env_preprocessor, env_postprocessor, preprocessor, postprocessor,
                     seeds=None, return_observations=False, render_callback=None):
    policy.reset()
    observation, info = env.reset(seed=seeds)
    if render_callback is not None:
        render_callback(env)
    all_observations, all_actions, all_rewards, all_successes, all_dones = [], [], [], [], []
    step = 0
    done = np.array([False] * env.num_envs)
    max_steps = env.call("_max_episode_steps")[0]
    progbar = trange(max_steps, desc=f"rollout <=({max_steps} steps)", disable=inside_slurm(), leave=False)
    check_env_attributes_and_types(env)
    while not np.all(done) and step < max_steps:
        observation = preprocess_observation(observation)
        observation = add_envs_task(env, observation)
        observation = env_preprocessor(observation)
        if return_observations:
            flat = {k: v for k, v in observation.items() if isinstance(v, torch.Tensor)}
            all_observations.append(deepcopy(flat))
        observation = preprocessor(observation)
        with torch.inference_mode():
            action = policy.select_action(observation)
        action = postprocessor(action)
        action_transition = {_ACTION: action}
        action_transition = env_postprocessor(action_transition)
        action = action_transition[_ACTION]
        action_numpy = action.to("cpu").numpy()
        observation, reward, terminated, truncated, info = env.step(action_numpy)
        if render_callback is not None:
            render_callback(env)
        successes = info["final_info"]["is_success"].tolist() if "final_info" in info else [False] * env.num_envs
        done = terminated | truncated | done
        if step + 1 == max_steps:
            done = np.ones_like(done, dtype=bool)
        all_actions.append(torch.from_numpy(action_numpy))
        all_rewards.append(torch.from_numpy(reward))
        all_dones.append(torch.from_numpy(done))
        all_successes.append(torch.tensor(successes))
        step += 1
        progbar.update()
    if return_observations:
        observation = preprocess_observation(observation)
        observation = add_envs_task(env, observation)
        observation = env_preprocessor(observation)
        flat = {k: v for k, v in observation.items() if isinstance(v, torch.Tensor)}
        all_observations.append(deepcopy(flat))
    ret = {
        _ACTION: torch.stack(all_actions, dim=1),
        "reward": torch.stack(all_rewards, dim=1),
        "success": torch.stack(all_successes, dim=1),
        "done": torch.stack(all_dones, dim=1),
    }
    if return_observations:
        stacked = {}
        for key in all_observations[0]:
            stacked[key] = torch.stack([obs[key] for obs in all_observations], dim=1)
        ret[OBS_STR] = stacked
    if hasattr(policy, "use_original_modules"):
        policy.use_original_modules()
    return ret

_le.rollout = _patched_rollout


from lerobot.envs.configs import LiberoEnv as LiberoEnvCfg
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.factory import make_pre_post_processors
from lerobot.scripts.lerobot_eval import eval_policy

_env_cfg_tmp = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=[0])
env_preprocessor, env_postprocessor = make_env_pre_post_processors(
    env_cfg=_env_cfg_tmp, policy_cfg=policy.config)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=BC_POLICY_REPO,
    preprocessor_overrides={"device_processor": {"device": str(policy.config.device)}})
print("Env + preprocessors ready.")


policy_preprocessor.json: 0.00B [00:00, ?B/s]

policy_preprocessor_step_5_normalizer_pr(…):   0%|          | 0.00/416 [00:00<?, ?B/s]

policy_postprocessor.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Env + preprocessors ready.


## Step 12 — Smoke test: A_null on task 0

A_null uses the zero-initialized null embedding, so injecting it is a no-op (adds a zero vector).
The policy should behave like the original BC baseline (~80% on task 0).

If A_null gives 0/3, training broke the policy's base capability. Stop and diagnose.
If A_null gives 2/3 or 3/3, training is healthy. Proceed to full eval.


In [ ]:
# Clear the stale training label so inference uses eval label
if hasattr(policy.model, "_train_advantage_label"):
    del policy.model._train_advantage_label

# Verify
print("_train_advantage_label cleared:", not hasattr(policy.model, "_train_advantage_label"))
print("_eval_advantage_label:", getattr(policy.model, "_eval_advantage_label", "NOT SET"))

# Set eval advantage
set_eval_advantage(policy, ADV_NULL)
print("Set to:", policy.model._eval_advantage_label)

In [ ]:
env_cfg_t0 = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=[0])
envs_t0 = make_env(env_cfg_t0, n_envs=1, use_async_envs=False)
env = envs_t0[LIBERO_SUITE][0]

set_eval_advantage(policy, ADV_NULL)
policy.eval()
t0 = time.time()
result = eval_policy(
    env=env, policy=policy,
    env_preprocessor=env_preprocessor, env_postprocessor=env_postprocessor,
    preprocessor=preprocessor, postprocessor=postprocessor,
    n_episodes=3, max_episodes_rendered=0, videos_dir=None,
    return_episode_data=False, start_seed=40000,
)
successes = [bool(ep["success"]) for ep in result["per_episode"]]
print(f"A_null task 0: {sum(successes)}/3  ({(time.time()-t0)/60:.1f} min)")
print("Expected ~2/3 or 3/3 if training is healthy.")


## Step 13 — Full eval: three conditions × 10 tasks × 5 episodes

~4 hours of eval time. Outputs a table comparing A_pos / A_neg / A_null with per-task breakdown.


In [ ]:
import json

N_EPISODES_PER_TASK = 5
TASK_IDS = list(range(10))

env_cfg_full = LiberoEnvCfg(task=LIBERO_SUITE, task_ids=TASK_IDS)
envs_by_suite = make_env(env_cfg_full, n_envs=1, use_async_envs=False)
task_envs = envs_by_suite[LIBERO_SUITE]


def run_condition(label_name, label_value):
    print(f"\n{'='*60}")
    print(f"Condition: {label_name} (advantage={label_value})")
    print(f"{'='*60}")
    set_eval_advantage(policy, label_value)
    policy.eval()
    per_task = {}
    total_s = total_n = 0
    t_cond = time.time()
    for task_id in TASK_IDS:
        env = task_envs[task_id]
        result = eval_policy(
            env=env, policy=policy,
            env_preprocessor=env_preprocessor, env_postprocessor=env_postprocessor,
            preprocessor=preprocessor, postprocessor=postprocessor,
            n_episodes=N_EPISODES_PER_TASK,
            max_episodes_rendered=0, videos_dir=None,
            return_episode_data=False,
            start_seed=10000 + 1000 * task_id,
        )
        successes = [bool(ep["success"]) for ep in result["per_episode"]]
        s = sum(successes); n = len(successes)
        per_task[task_id] = successes
        total_s += s; total_n += n
        print(f"  task {task_id}: {s}/{n}  ({100*s/n:.0f}%)")
    elapsed = (time.time() - t_cond) / 60
    print(f"\n  OVERALL {label_name}: {total_s}/{total_n} = {100*total_s/total_n:.1f}%  ({elapsed:.1f} min)")
    return per_task, (total_s, total_n)


t0 = time.time()
pos_per_task,  pos_overall  = run_condition("A_pos",  ADV_POS)
neg_per_task,  neg_overall  = run_condition("A_neg",  ADV_NEG)
null_per_task, null_overall = run_condition("A_null", ADV_NULL)

print("\n\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Total eval time: {(time.time()-t0)/60:.1f} min\n")
print(f"{'Task':<6} {'A_pos':<10} {'A_neg':<10} {'A_null':<10}")
print("-" * 40)
for task_id in TASK_IDS:
    pos_s = sum(pos_per_task[task_id]); neg_s = sum(neg_per_task[task_id]); null_s = sum(null_per_task[task_id])
    print(f"{task_id:<6} {pos_s}/{N_EPISODES_PER_TASK:<8} {neg_s}/{N_EPISODES_PER_TASK:<8} {null_s}/{N_EPISODES_PER_TASK:<8}")
print("-" * 40)
print(f"{'TOT':<6} {pos_overall[0]}/{pos_overall[1]:<8} {neg_overall[0]}/{neg_overall[1]:<8} {null_overall[0]}/{null_overall[1]:<8}")
print(f"{'%':<6} {100*pos_overall[0]/pos_overall[1]:<9.1f}% {100*neg_overall[0]/neg_overall[1]:<9.1f}% {100*null_overall[0]/null_overall[1]:<9.1f}%")

results = {
    "n_episodes_per_task": N_EPISODES_PER_TASK,
    "A_pos":  {"per_task": {str(k): v for k, v in pos_per_task.items()},  "overall": list(pos_overall)},
    "A_neg":  {"per_task": {str(k): v for k, v in neg_per_task.items()},  "overall": list(neg_overall)},
    "A_null": {"per_task": {str(k): v for k, v in null_per_task.items()}, "overall": list(null_overall)},
}
with open(RECAP_V2_DIR / "eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved results to {RECAP_V2_DIR / 'eval_results.json'}")
